# Chapter 02. 단어 빈도 분석과 Word Cloud 만들기

Chapter 01에서 만든 `book_bestseller_clean.csv`를 이용해 **도서 제목에 어떤 단어가 자주 등장하는지** 확인합니다.

이번 Chapter의 흐름은 다음과 같습니다.

**전처리 파일 불러오기 → 상품명 확인 → Dictionary 빈도 누적 → Counter → value_counts() → 단순 토큰화 → Kiwi 형태소 분석 → 명사 추출 → 길이 필터 → 불용어 → 상위 30개 → 막대그래프 → Word Cloud → 원본 제목 검증 → Markdown 정리**

이번에도 **해야 할 일 이해 → AI에게 질문 → 코드 실행 → 결과 확인 → 검증 → Markdown 정리** 순서로 진행합니다.

> 먼저 Chapter 01 Notebook을 끝까지 실행해서 `notebooks/book-text-ml/book_bestseller_clean.csv`가 만들어져 있어야 합니다.

## 실습 1. 전처리 데이터 불러오기

### AI에게 질문

> Python과 pandas를 처음 배우고 있습니다.  
> Chapter 01에서 만든 `book_bestseller_clean.csv` 파일을 pandas로 `df_books`라는 DataFrame에 불러오고 싶습니다.  
> 다음 내용을 확인하는 가장 간단한 코드를 작성해 주세요.
>
> 1. 전체 행과 컬럼 개수  
> 2. 컬럼 이름  
> 3. 앞의 5행  
> 4. `상품명` 컬럼이 존재하는지 확인  
>
> CSV는 `utf-8-sig` 인코딩을 사용했습니다.  
> 초보자가 이해하기 쉽도록 불필요하게 복잡한 코드는 사용하지 말아 주세요.

### AI 답변

`pd.read_csv()`로 CSV 파일을 읽은 뒤 `shape`, `columns`, `head()`를 이용해 데이터가 정상적으로 연결되었는지 확인하면 됩니다.  
`"상품명" in df_books.columns`은 상품명 컬럼이 실제로 존재하는지 `True / False`로 확인하는 코드입니다.

In [ ]:
# pandas를 불러옵니다.
import pandas as pd

# Chapter 01에서 만든 전처리 CSV 파일 위치입니다.
DATA_PATH = "notebooks/book-text-ml/book_bestseller_clean.csv"

# CSV 파일을 df_books라는 DataFrame으로 불러옵니다.
df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

# 전체 행과 컬럼 개수를 확인합니다.
print("데이터 크기:", df_books.shape)

# 컬럼 이름을 확인합니다.
print("컬럼:", df_books.columns.tolist())

# '상품명' 컬럼이 실제로 있는지 확인합니다.
print("'상품명' 컬럼 존재:", "상품명" in df_books.columns)

# 앞의 5행을 확인합니다.
df_books.head()

### 실습 1 결과 확인 및 정리

이 단계에서는 바로 단어 분석을 시작하지 않고 **Chapter 01에서 만든 파일이 정상적으로 연결되는지 먼저 확인**했습니다.

특히 다음 네 가지를 확인합니다.

- CSV 파일이 오류 없이 열리는가?
- Chapter 01에서 저장했던 데이터와 행 개수가 크게 달라지지 않았는가?
- `상품명` 컬럼이 존재하는가?
- 한글 제목이 깨지지 않고 정상적으로 보이는가?

파일을 찾지 못한다면 단어 분석 코드를 고치기 전에 먼저 **현재 작업 폴더와 파일 경로**를 확인해야 합니다.

## 실습 2. 분석할 도서 제목 확인하기

단어를 세기 전에 `상품명` 컬럼에 실제로 어떤 제목들이 들어 있는지 먼저 확인합니다.

도서 제목에는 한글, 영어, 숫자, 괄호, 특수문자, 부제, 개정판·세트 표시 등이 함께 들어 있을 수 있습니다.  
따라서 처음부터 복잡한 형태소 분석으로 넘어가기보다 먼저 제목 데이터를 눈으로 확인합니다.

In [ ]:
# 상품명 앞의 20개를 확인합니다.
print(df_books["상품명"].head(20))

# 상품명 컬럼의 결측치 개수를 확인합니다.
print("\n상품명 결측치:", df_books["상품명"].isna().sum())

# 결측치는 빈 문자열로 바꾸고 문자열로 변환한 뒤 앞뒤 공백을 제거합니다.
titles = (
    df_books["상품명"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# 제목이 완전히 빈 행은 제외합니다.
titles = titles[titles != ""]

# 실제 분석에 사용할 제목 개수를 확인합니다.
print("사용할 제목 개수:", len(titles))

titles.head()

### 실습 2 결과 확인 및 정리

`titles`는 앞으로 단어 분석에 사용할 **정리된 도서 제목 Series**입니다.

처리 순서는 다음과 같습니다.

1. 상품명 결측치를 빈 문자열로 바꿈
2. 문자열 자료형으로 변환
3. 제목 앞뒤 공백 제거
4. 완전히 비어 있는 제목 제외

이렇게 제목을 먼저 정리해 두면 뒤의 반복문과 형태소 분석 코드가 조금 더 단순해집니다.

## 실습 3. Dictionary로 단어 빈도 원리 이해하기

빈도 분석의 핵심은 **같은 단어가 등장할 때마다 해당 단어의 숫자를 1씩 증가시키는 것**입니다.

먼저 실제 도서 제목이 아니라 아주 작은 예시로 원리를 확인합니다.

In [ ]:
# 빈도 계산 원리를 이해하기 위한 예시 단어 목록입니다.
sample_words = [
    "데이터",
    "분석",
    "파이썬",
    "데이터",
    "AI",
    "분석",
    "데이터",
]

# 단어별 등장 횟수를 저장할 빈 Dictionary를 만듭니다.
word_counts = {}

# 단어를 하나씩 꺼냅니다.
for word in sample_words:
    # 이미 등장한 단어라면 기존 숫자에 1을 더합니다.
    if word in word_counts:
        word_counts[word] += 1
    # 처음 등장한 단어라면 1부터 시작합니다.
    else:
        word_counts[word] = 1

print(word_counts)

### 실습 3 결과 확인 및 정리

Dictionary 방식에서는 빈도 계산 과정을 직접 눈으로 볼 수 있습니다.

- 처음에는 `word_counts = {}`로 빈 Dictionary를 만듭니다.
- 단어를 하나씩 확인합니다.
- 이미 Dictionary에 있으면 `+ 1`을 합니다.
- 처음 등장한 단어라면 `1`로 저장합니다.

즉 핵심 흐름은 **단어 하나 확인 → 이전 등장 여부 확인 → 횟수 1 증가 → 반복**입니다.

같은 코드는 `get()`을 사용해 더 짧게 쓸 수도 있습니다.

In [ ]:
# get(word, 0)은 단어가 아직 없으면 0을 사용한다는 뜻입니다.
word_counts_get = {}

for word in sample_words:
    word_counts_get[word] = word_counts_get.get(word, 0) + 1

print(word_counts_get)

## 실습 4. Counter로 같은 작업 간단하게 처리하기

Dictionary로 원리를 이해했으니 이번에는 Python 표준 라이브러리의 `Counter`를 사용합니다.

`Counter`는 리스트 안에서 같은 값이 몇 번 등장했는지 자동으로 세어 줍니다.

In [ ]:
# Counter를 불러옵니다.
from collections import Counter

# sample_words의 단어 빈도를 자동으로 계산합니다.
sample_counter = Counter(sample_words)

print(sample_counter)

# 전체 빈도를 높은 순서대로 확인합니다.
print(sample_counter.most_common())

# 상위 3개만 확인합니다.
print(sample_counter.most_common(3))

### 실습 4 결과 확인 및 정리

Dictionary와 Counter는 완전히 다른 분석을 하는 것이 아닙니다.

- **Dictionary** → 빈도가 어떻게 누적되는지 원리를 직접 확인
- **Counter** → 같은 작업을 훨씬 간단하게 수행

`most_common(3)`처럼 숫자를 넣으면 가장 많이 등장한 단어부터 원하는 개수만 확인할 수 있습니다.

## 실습 5. pandas value_counts()로 빈도 확인하기

pandas의 Series에도 같은 값의 등장 횟수를 세는 `value_counts()`가 있습니다.

In [ ]:
# sample_words 리스트를 pandas Series로 바꿉니다.
sample_series = pd.Series(sample_words)

# 각 단어가 몇 번 등장했는지 확인합니다.
sample_series.value_counts()

### 실습 5 결과 확인 및 정리

이번까지 세 가지 빈도 집계 방법을 확인했습니다.

- **Dictionary** → 직접 빈도를 누적해 원리를 이해하기 좋음
- **Counter** → Python에서 간단하게 빈도 계산
- **value_counts()** → pandas Series에서 간단하게 빈도 계산

어느 하나만 정답인 것은 아닙니다. 데이터 형태와 상황에 따라 편한 방법을 사용할 수 있습니다.

## 실습 6. 실제 도서 제목을 간단하게 단어로 나누기

한글 형태소 분석을 사용하기 전에 정규표현식으로 간단하게 토큰을 나눠 봅니다.

이번 단계의 목적은 **완벽한 단어 추출이 아니라 실제 제목 데이터에서도 빈도 집계 흐름을 연결해 보는 것**입니다.

In [ ]:
# 정규표현식을 사용하기 위해 re를 불러옵니다.
import re

# 한글, 영문, 숫자가 연속된 부분을 하나의 토큰으로 추출합니다.
def simple_tokenize(text):
    return re.findall(r"[가-힣A-Za-z0-9]+", str(text).lower())

# 간단한 문장으로 결과를 확인합니다.
sample_title = "AI 시대의 데이터 분석, Python으로 시작하기!"

simple_tokenize(sample_title)

### 실습 6 결과 확인 및 정리

단순 토큰화는 특수문자를 제거하고 한글·영문·숫자 덩어리를 비교적 쉽게 나눌 수 있습니다.

하지만 `시대의`, `python으로`, `시작하기`처럼 **조사나 어미가 붙은 형태가 그대로 남을 수 있습니다.**

즉 사람이 생각하는 핵심 단어와 정확히 일치하지 않을 수 있습니다.  
이 한계 때문에 뒤에서 Kiwi 형태소 분석기를 사용합니다.

## 실습 7. 모든 제목에서 단어 추출하기

앞에서 만든 `simple_tokenize()`를 모든 제목에 적용해 단어를 하나의 리스트에 모읍니다.

In [ ]:
# 모든 제목에서 추출한 단어를 저장할 빈 리스트입니다.
simple_words = []

# 제목을 하나씩 가져옵니다.
for title in titles:
    # 한 제목을 단어 리스트로 나눕니다.
    words = simple_tokenize(title)

    # extend()로 단어들을 기존 리스트에 이어 붙입니다.
    simple_words.extend(words)

print("전체 단어 수:", len(simple_words))
print(simple_words[:50])

### 실습 7 결과 확인 및 정리

여기서는 `append()`가 아니라 `extend()`를 사용했습니다.

- `append(["데이터", "분석"])` → 리스트 안에 리스트가 들어감
- `extend(["데이터", "분석"])` → "데이터", "분석"이 기존 리스트에 각각 추가됨

빈도 분석에서는 모든 단어가 하나의 리스트에 이어져 있는 형태가 사용하기 편합니다.

## 실습 8. 실제 데이터에서 Counter로 빈도 확인하기

이제 실제 제목에서 추출한 `simple_words`의 빈도를 계산합니다.

In [ ]:
# 실제 제목에서 추출한 단어들의 빈도를 계산합니다.
simple_counter = Counter(simple_words)

# 상위 30개를 확인합니다.
print(simple_counter.most_common(30))

# 표 형태로 보기 위해 DataFrame으로 바꿉니다.
simple_top30 = pd.DataFrame(
    simple_counter.most_common(30),
    columns=["단어", "빈도"]
)

simple_top30

### 실습 8 결과 확인 및 정리

아직 Word Cloud를 만들지 않고 먼저 상위 단어를 눈으로 확인합니다.

확인할 내용은 다음과 같습니다.

- 의미 없는 한 글자 단어가 많은가?
- 조사나 어미가 붙은 단어가 보이는가?
- 같은 의미인데 서로 다른 형태로 나뉘었는가?
- 숫자가 지나치게 많이 등장하는가?
- 세트·개정판 같은 분석 목적과 관계가 적은 표현이 많은가?

이 결과를 보고 뒤의 형태소 분석과 필터 기준을 정합니다.

## 실습 9. pandas로도 실제 단어 빈도 확인하기

앞에서 만든 `simple_words`를 Series로 바꾼 뒤 `value_counts()` 결과를 Counter와 비교합니다.

In [ ]:
# 단어 리스트를 pandas Series로 바꿉니다.
simple_word_series = pd.Series(simple_words)

# pandas로 상위 30개를 확인합니다.
display(simple_word_series.value_counts().head(30))

# Counter와 pandas의 상위 10개 결과를 함께 출력합니다.
print("Counter 상위 10개")
print(simple_counter.most_common(10))

print("\npandas value_counts 상위 10개")
print(simple_word_series.value_counts().head(10))

### 실습 9 결과 확인 및 정리

같은 `simple_words`를 사용했다면 Counter와 `value_counts()`의 상위 단어와 빈도는 같은 방향으로 나와야 합니다.

두 결과가 다르다면 바로 다음 단계로 넘어가기보다 **simple_words가 만들어지는 과정부터 다시 확인**해야 합니다.

## 실습 10. 왜 형태소 분석이 필요한가?

한국어는 단순히 공백만 기준으로 나누기 어렵습니다.

예를 들어 다음 표현은 사람이 볼 때 모두 핵심 단어가 **데이터**입니다.

- 데이터를
- 데이터가
- 데이터의
- 데이터로

하지만 단순 문자열 분리에서는 서로 다른 값이 될 수 있습니다.

또한 `은`, `는`, `이`, `가`, `을`, `를`, `의` 같은 요소도 함께 나타날 수 있습니다.

이번 분석에서는 도서 제목의 주요 주제를 보기 위해 **명사 중심으로 단어를 추출**합니다.

## 실습 11. Kiwi 형태소 분석기 설치하기

### AI에게 질문

> Windows와 VS Code Notebook 환경에서 Python을 배우고 있습니다.  
> 한국어 도서 제목의 명사를 추출하기 위해 `kiwipiepy`를 설치하려고 합니다.  
> 1. 현재 가상환경에 설치하는 명령  
> 2. 설치 확인 명령  
> 3. ModuleNotFoundError가 발생할 때 확인할 항목  
> 을 초보자가 따라할 수 있게 순서대로 설명해 주세요.

### AI 답변

VS Code 터미널에서 현재 Notebook이 사용하는 Python 환경과 같은 환경을 선택한 뒤 설치하는 것이 중요합니다.

터미널에서 다음 명령을 사용합니다.

```powershell
python -m pip install kiwipiepy
```

설치 확인:

```powershell
python -m pip show kiwipiepy
```

설치했는데 Notebook에서 `ModuleNotFoundError`가 발생하면 **VS Code에서 선택한 Notebook 커널과 패키지를 설치한 Python 환경이 같은지** 확인하고, 필요하면 커널을 재시작합니다.

In [ ]:
# Notebook에서 설치 여부만 확인하고 싶다면 아래 import를 실행합니다.
# 설치되지 않았다면 위의 터미널 명령으로 먼저 설치하세요.
from kiwipiepy import Kiwi

print("Kiwi import 성공")

## 실습 12. Kiwi로 한 문장 품사 확인하기

설치 후 바로 전체 데이터에 적용하지 않고 짧은 예시 문장으로 형태소 분석 결과를 확인합니다.

In [ ]:
# Kiwi 형태소 분석기를 만듭니다.
kiwi = Kiwi()

# 테스트할 문장입니다.
sample_title = "AI 시대의 데이터 분석을 위한 파이썬"

# 문장을 형태소 단위로 분석합니다.
tokens = kiwi.tokenize(sample_title)

# 전체 토큰 정보를 확인합니다.
tokens

In [ ]:
# 각 형태소의 실제 글자(form)와 품사 태그(tag)를 확인합니다.
for token in tokens:
    print(token.form, token.tag)

### 실습 12 결과 확인 및 정리

이번 실습에서 주로 사용할 품사 태그는 다음과 같습니다.

- `NNG` → 일반 명사
- `NNP` → 고유 명사
- `SL` → 알파벳으로 된 외국어

형태소 분석 결과도 절대적인 정답은 아닙니다. 도서 제목에는 신조어, 브랜드명, 사람 이름, 영문 약어 등이 많기 때문에 **실제 분석 결과를 직접 확인해야 합니다.**

## 실습 13. 명사 중심으로 단어 추출하기

사용할 품사만 선택하는 함수를 만듭니다.

In [ ]:
# 이번 분석에서 사용할 품사 태그입니다.
TARGET_TAGS = {"NNG", "NNP", "SL"}

def extract_words(text):
    # 추출한 단어를 담을 빈 리스트입니다.
    result = []

    # 문장을 Kiwi로 형태소 분석합니다.
    for token in kiwi.tokenize(str(text)):
        # 일반 명사, 고유 명사, 영문 토큰만 사용합니다.
        if token.tag in TARGET_TAGS:
            # 앞뒤 공백을 제거하고 영문 대소문자를 소문자로 통일합니다.
            word = token.form.strip().lower()
            result.append(word)

    return result

# 간단한 문장으로 함수를 테스트합니다.
extract_words("AI 시대의 데이터 분석을 위한 파이썬")

### 실습 13 결과 확인 및 정리

함수의 흐름은 다음과 같습니다.

**문장 분석 → 형태소 하나씩 확인 → NNG·NNP·SL만 선택 → 공백 제거 → 소문자 통일 → 리스트 반환**

영문을 소문자로 통일하는 이유는 `AI`와 `ai`처럼 대소문자만 다른 표현이 서로 다른 단어로 집계되는 것을 줄이기 위해서입니다.

## 실습 14. 최소 글자수 필터링 적용하기

한 글자 단어가 모두 의미 없는 것은 아니지만, 이번 제목 빈도 분석에서는 노이즈를 줄이기 위해 **2글자 이상**을 기본 조건으로 사용합니다.

In [ ]:
# 최소 단어 길이를 2글자로 정합니다.
MIN_LENGTH = 2

def extract_words(text):
    result = []

    for token in kiwi.tokenize(str(text)):
        # 사용할 품사만 통과시킵니다.
        if token.tag in TARGET_TAGS:
            word = token.form.strip().lower()

            # 2글자 이상인 단어만 추가합니다.
            if len(word) >= MIN_LENGTH:
                result.append(word)

    return result

extract_words("AI 시대의 데이터 분석을 위한 파이썬")

### 실습 14 결과 확인 및 정리

`MIN_LENGTH = 2`는 자연 법칙이 아니라 **이번 분석을 위해 정한 조건**입니다.

따라서 보고서에는 “이번 분석에서는 노이즈를 줄이기 위해 2글자 이상의 명사와 영문 토큰을 사용했다”처럼 조건을 기록하는 것이 좋습니다.

영문 `ai`는 2글자이므로 포함됩니다.

## 실습 15. 모든 도서 제목에서 명사 추출하기

이제 현재 `extract_words()` 함수를 모든 제목에 적용합니다.

In [ ]:
# 형태소 분석으로 추출한 모든 단어를 저장합니다.
filtered_words = []

for title in titles:
    words = extract_words(title)
    filtered_words.extend(words)

print("추출된 전체 단어 수:", len(filtered_words))
print(filtered_words[:50])

# 빈도를 계산합니다.
noun_counter = Counter(filtered_words)

# 상위 30개를 DataFrame으로 만듭니다.
noun_top30 = pd.DataFrame(
    noun_counter.most_common(30),
    columns=["단어", "빈도"]
)

noun_top30

In [ ]:
# 단순 토큰화 결과와 형태소 분석 결과를 비교합니다.
print("단순 토큰화 상위 단어")
display(simple_top30.head(10))

print("형태소 분석 후 상위 단어")
display(noun_top30.head(10))

### 실습 15 결과 확인 및 정리

단순 토큰화와 형태소 분석 결과를 비교하면서 다음을 확인합니다.

- 조사나 어미가 붙은 표현이 줄었는가?
- 의미 있는 주제어가 더 잘 보이는가?
- 반대로 중요한 단어가 사라진 것은 없는가?

형태소 분석기를 사용했다고 해서 무조건 결과가 좋아지는 것은 아니므로 **전후 비교가 필요합니다.**

## 실습 16. 불용어 이해하기

형태소 분석 뒤에도 분석 목적과 관계가 적은 표현이 상위에 나타날 수 있습니다.

예를 들어 실제 결과를 확인한 뒤 다음과 같은 표현이 반복될 수 있습니다.

- 세트
- 개정판
- 에디션
- 리커버
- 한정판

이처럼 자주 등장하지만 분석 목적에는 큰 도움이 되지 않는 단어를 **불용어(stopword)** 로 지정할 수 있습니다.

하지만 결과를 보기도 전에 많은 단어를 임의로 제거하면 분석자가 원하는 결과만 남길 위험이 있습니다.

따라서 **빈도 결과 확인 → 원본 제목 확인 → 제거 이유 판단 → 불용어 추가 → 다시 결과 확인** 순서가 중요합니다.

## 실습 17. 불용어 목록 만들기

처음부터 너무 많은 단어를 제거하지 않고, 수업 예시의 기본 불용어만 적용합니다.

In [ ]:
# 사용할 품사
TARGET_TAGS = {"NNG", "NNP", "SL"}

# 최소 단어 길이
MIN_LENGTH = 2

# 분석 목적과 관계가 적은 표현 예시
STOPWORDS = {
    "세트",
    "개정판",
    "에디션",
    "리커버",
    "한정판",
}

def extract_words(text):
    result = []

    for token in kiwi.tokenize(str(text)):
        # 1. 사용할 품사가 아니면 건너뜁니다.
        if token.tag not in TARGET_TAGS:
            continue

        word = token.form.strip().lower()

        # 2. 2글자 미만이면 건너뜁니다.
        if len(word) < MIN_LENGTH:
            continue

        # 3. 불용어 목록에 있으면 건너뜁니다.
        if word in STOPWORDS:
            continue

        result.append(word)

    return result

extract_words("AI 시대의 데이터 분석 개정판 세트")

### 실습 17 결과 확인 및 정리

최종 단어 추출 함수에는 세 가지 필터가 들어 있습니다.

1. **품사 필터** → NNG, NNP, SL
2. **길이 필터** → 2글자 이상
3. **불용어 필터** → STOPWORDS에 없는 단어

불용어는 결과에 직접 영향을 주는 조건이므로 나중에 어떤 단어를 제거했는지 기록해야 합니다.

## 실습 18. 최종 단어 목록 만들기

불용어까지 적용한 최종 함수를 전체 제목에 적용하고 상위 30개 단어를 만듭니다.

In [ ]:
# 최종 단어를 모두 모읍니다.
final_words = []

for title in titles:
    final_words.extend(extract_words(title))

print("최종 단어 수:", len(final_words))
print(final_words[:50])

# 최종 빈도를 계산합니다.
final_counter = Counter(final_words)

# 상위 30개 단어를 DataFrame으로 만듭니다.
top30 = pd.DataFrame(
    final_counter.most_common(30),
    columns=["단어", "빈도"]
)

top30

In [ ]:
# 상위 30개 결과를 CSV로 저장합니다.
TOP30_PATH = "notebooks/book-text-ml/chapter02_top30_words.csv"

top30.to_csv(
    TOP30_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", TOP30_PATH)

### 실습 18 결과 확인 및 정리

이 단계에서 `top30`이 이번 분석의 핵심 빈도표입니다.

또한 `chapter02_top30_words.csv`로 저장했기 때문에 Notebook 밖에서도 Excel 등으로 결과를 확인할 수 있습니다.

상위 30개 단어는 Word Cloud를 만들기 전에 반드시 표로 먼저 확인합니다.

## 실습 19. 상위 단어를 막대그래프로 먼저 확인하기

Word Cloud는 전체 분위기를 보기에는 좋지만 정확한 빈도 차이를 비교하기에는 막대그래프가 더 편합니다.

In [ ]:
# 그래프를 그리기 위해 matplotlib를 불러옵니다.
import matplotlib.pyplot as plt

# Windows에서 한글이 깨지지 않도록 맑은 고딕을 사용합니다.
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 가장 높은 빈도가 위쪽에 오도록 정렬합니다.
plot_data = top30.sort_values("빈도", ascending=True)

# 가로 막대그래프를 그립니다.
plt.figure(figsize=(10, 8))
plt.barh(plot_data["단어"], plot_data["빈도"])
plt.xlabel("빈도")
plt.ylabel("단어")
plt.title("베스트셀러 도서 제목 상위 30개 단어")
plt.tight_layout()
plt.show()

### 실습 19 결과 확인 및 정리

그래프에서 다음을 확인합니다.

- 한글이 깨지지 않는가?
- 표에서 가장 빈도가 높은 단어가 그래프에서도 가장 긴 막대인가?
- 막대 길이와 빈도 순서가 자연스럽게 대응하는가?

Word Cloud보다 막대그래프가 **정확한 순위와 빈도 차이**를 비교하기에는 더 좋습니다.

## 실습 20. Word Cloud 설치하기

Word Cloud 라이브러리가 설치되어 있지 않다면 VS Code 터미널에서 다음 명령을 실행합니다.

```powershell
python -m pip install wordcloud matplotlib
```

설치 확인:

```powershell
python -m pip show wordcloud
```

Notebook에서 아래 import가 오류 없이 실행되면 사용할 수 있습니다.

In [ ]:
# Word Cloud 라이브러리가 설치되어 있는지 확인합니다.
from wordcloud import WordCloud

print("WordCloud import 성공")

## 실습 21. Word Cloud용 한글 폰트 확인하기

Word Cloud는 Matplotlib와 달리 **한글 폰트 파일의 실제 경로**를 지정해야 합니다.

Windows의 맑은 고딕 파일이 있는지 확인합니다.

In [ ]:
# 파일 존재 여부 확인에 Path를 사용합니다.
from pathlib import Path

# Windows 맑은 고딕 폰트의 일반적인 경로입니다.
FONT_PATH = Path("C:/Windows/Fonts/malgun.ttf")

print("폰트 경로:", FONT_PATH)
print("폰트 존재 여부:", FONT_PATH.exists())

### 실습 21 결과 확인 및 정리

`True`가 나오면 해당 폰트를 Word Cloud에 사용할 수 있습니다.

`False`라면 PC에 설치된 다른 한글 폰트 파일의 실제 경로를 확인해야 합니다.  
폰트 경로는 운영체제와 컴퓨터 환경에 따라 다를 수 있습니다.

## 실습 22. Word Cloud 생성하기

최종 빈도 정보인 `final_counter`를 그대로 Word Cloud에 사용합니다.

In [ ]:
# 폰트 파일이 없으면 Word Cloud 생성 전에 오류를 알려 줍니다.
if not FONT_PATH.exists():
    raise FileNotFoundError(f"한글 폰트 파일을 찾을 수 없습니다: {FONT_PATH}")

# 단어 빈도를 이용해 Word Cloud를 만듭니다.
wordcloud = WordCloud(
    font_path=str(FONT_PATH),
    width=1200,
    height=700,
    background_color="white",
    max_words=100,
    random_state=42,
).generate_from_frequencies(final_counter)

# Notebook 화면에 표시합니다.
plt.figure(figsize=(12, 7))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("교보문고 베스트셀러 제목 Word Cloud")
plt.tight_layout()
plt.show()

# 이미지 파일로 저장합니다.
WORDCLOUD_PATH = "notebooks/book-text-ml/chapter02_wordcloud.png"
wordcloud.to_file(WORDCLOUD_PATH)

print("저장 완료:", WORDCLOUD_PATH)

### 실습 22 결과 확인 및 정리

Word Cloud에서 글자가 크게 보인다는 것은 **이번 분석 조건에서 그 단어가 더 자주 등장했다는 뜻**입니다.

또한 `random_state=42`를 지정해 같은 데이터로 다시 실행했을 때 배치가 크게 달라지는 것을 줄였습니다.

최종 이미지는 `chapter02_wordcloud.png`로 저장했습니다.

## 실습 23. Word Cloud를 어떻게 해석해야 할까?

Word Cloud는 **도서 제목에서 단어가 얼마나 자주 등장했는지**를 시각적으로 보여 줍니다.

예를 들어 어떤 단어가 크게 보이면 다음 정도로 해석할 수 있습니다.

> 분석에 사용한 베스트셀러 도서 제목에서 해당 단어가 상대적으로 자주 등장했다.

하지만 다음처럼 확대 해석하면 안 됩니다.

- 그 단어가 도서 판매의 원인이다.
- 독자들이 그 주제를 가장 좋아한다.

이번 분석은 **제목의 단어 빈도**를 보는 것이므로, 단어의 등장 횟수와 판매 원인은 서로 다른 개념입니다.

## 실습 24. 상위 단어를 원본 제목에서 다시 확인하기

빈도 결과가 실제 제목에서 어떻게 사용되었는지 원본 데이터로 돌아가 확인합니다.

In [ ]:
# 상위 30개 중 가장 빈도가 높은 첫 번째 단어를 가져옵니다.
target_word = top30.iloc[0]["단어"]

print("확인할 단어:", target_word)

# 원본 상품명에 해당 문자열이 포함된 제목을 찾습니다.
matched_titles = df_books[
    df_books["상품명"].astype(str).str.contains(
        str(target_word),
        case=False,
        na=False,
        regex=False,
    )
][["상품명", "분야"]]

matched_titles.head(20)

### 실습 24 결과 확인 및 정리

형태소 분석에서 같은 단어로 집계되더라도 실제 제목에서는 서로 다른 문맥으로 사용될 수 있습니다.

또한 형태소 분석 결과와 단순 문자열 검색의 개수는 정확히 일치하지 않을 수 있습니다.  
합성어의 일부로 분석되거나 형태가 변형되는 경우가 있기 때문입니다.

따라서 이 단계는 **빈도 결과의 의미를 원본 제목에서 다시 확인하는 검증 과정**입니다.

## 실습 25. 상위 10개 단어의 빈도 합 검증하기

분석 결과가 만들어졌다고 바로 끝내지 않고 일부 결과를 서로 비교합니다.

In [ ]:
# Counter에서 직접 상위 10개를 가져옵니다.
top10 = final_counter.most_common(10)

print("Counter Top 10")
print(top10)

print("\nDataFrame Top 10")
print(top30.head(10).values.tolist())

# 전체 빈도 합과 최종 단어 리스트 길이는 같아야 합니다.
print("\nCounter 전체 빈도 합:", sum(final_counter.values()))
print("final_words 길이:", len(final_words))

### 실습 25 결과 확인 및 정리

같은 데이터에서 만든 결과라면

- Counter 상위 10개
- DataFrame 상위 10개

의 단어와 빈도가 같아야 합니다.

또한 `sum(final_counter.values())`와 `len(final_words)`도 같아야 합니다.

이런 검증은 **코드가 실행되었다는 사실**과 **분석 로직이 서로 맞는다는 사실**을 구분하는 데 도움이 됩니다.

## 실습 26. 불용어 적용 전후 비교하기

불용어를 적용하기 전 `noun_counter`와 적용 후 `final_counter`를 비교합니다.

In [ ]:
# 불용어 적용 전 상위 20개
before_top20 = pd.DataFrame(
    noun_counter.most_common(20),
    columns=["적용 전 단어", "적용 전 빈도"]
)

# 불용어 적용 후 상위 20개
after_top20 = pd.DataFrame(
    final_counter.most_common(20),
    columns=["적용 후 단어", "적용 후 빈도"]
)

# 두 표를 옆으로 붙여 비교합니다.
comparison = pd.concat(
    [before_top20, after_top20],
    axis=1,
)

comparison

### 실습 26 결과 확인 및 정리

비교표에서 다음을 확인합니다.

- 제거하려던 불용어가 실제로 사라졌는가?
- 중요한 단어까지 함께 제거된 것은 아닌가?
- 상위 순위가 어떻게 달라졌는가?
- 불용어를 추가한 이유를 설명할 수 있는가?

불용어는 분석 결과를 바꾸는 중요한 조건이므로 최종 Markdown에도 사용 기준을 기록해야 합니다.

## 실습 27. AI에게 결과 해석 요청하기

### AI에게 질문

실제 Notebook 실행 후 아래 `top30.head(10)` 결과를 보고 AI에게 해석 초안을 요청할 수 있습니다.

> 교보문고 베스트셀러 도서 제목에서 명사와 영문 토큰을 추출해 단어 빈도 분석을 했습니다.  
> 조건은 다음과 같습니다.
>
> - Kiwi 형태소 분석 사용
> - NNG, NNP, SL 품사 사용
> - 2글자 이상 사용
> - 불용어 제거
> - 빈도 상위 30개 확인
>
> 실제 상위 10개 결과는 Notebook에서 출력된 값을 사용합니다.  
> 이 결과를 초보자용 분석 보고서의 Markdown 문단으로 4~6문장 정도 작성해 주세요.
>
> 주의:
> 1. 단어 빈도만으로 판매 원인을 추정하지 말아 주세요.
> 2. 실제 숫자를 임의로 바꾸지 말아 주세요.
> 3. 관찰한 내용과 분석의 한계를 구분해 주세요.

### AI 답변 기준

AI에게 결과를 맡길 때는 실제 출력값을 제공해야 하며, 없는 숫자를 만들어 달라고 하면 안 됩니다.  
또한 “자주 등장했다”는 관찰과 “판매의 원인이다”라는 해석은 구분해야 합니다.

In [ ]:
# AI에게 전달할 실제 상위 10개 결과를 확인합니다.
top30.head(10)

## 실습 28. 결과 Markdown 작성하기

실제 실행된 `top30`의 상위 단어와 빈도를 사용해 Markdown을 자동으로 만듭니다.  
이렇게 하면 아직 실행하지 않은 숫자를 임의로 적지 않아도 됩니다.

In [ ]:
# Notebook에 Markdown을 표시하기 위해 불러옵니다.
from IPython.display import display, Markdown

# 실제 상위 3개 단어와 빈도를 가져옵니다.
word1, count1 = top30.iloc[0]["단어"], top30.iloc[0]["빈도"]
word2, count2 = top30.iloc[1]["단어"], top30.iloc[1]["빈도"]
word3, count3 = top30.iloc[2]["단어"], top30.iloc[2]["빈도"]

result_md = f"""
## 단어 빈도 분석 결과

교보문고 베스트셀러 도서 제목을 대상으로 단어 빈도를 확인했습니다.
Kiwi 형태소 분석기를 이용해 일반 명사(NNG), 고유 명사(NNP), 영문 토큰(SL)을 추출했고,
2글자 이상 단어를 사용한 뒤 분석 목적과 관계가 적은 일부 표현을 불용어로 제거했습니다.

실제 빈도 상위 3개 단어는 다음과 같습니다.

- 1위: **{word1}** ({count1}회)
- 2위: **{word2}** ({count2}회)
- 3위: **{word3}** ({count3}회)

Word Cloud에서도 빈도가 높은 단어가 상대적으로 크게 표시됩니다.
다만 이 결과는 도서 제목에서 자주 등장한 단어를 보여 주는 것이며,
해당 단어가 도서 판매의 원인이라는 의미는 아닙니다.
"""

display(Markdown(result_md))

### 실습 28 결과 확인 및 정리

이번 Markdown은 상위 단어와 빈도를 직접 입력하지 않고 `top30`의 실제 결과를 가져와 작성합니다.

따라서 **Kernel Restart → Run All**을 했을 때 실제 데이터 결과가 자동으로 반영됩니다.

분석 결과에서는 “많이 등장했다”는 사실까지만 말하고, 데이터가 지원하지 않는 판매 원인이나 독자 선호까지 확대 해석하지 않습니다.

## 실습 29. 전체 분석 흐름을 한 번에 다시 확인하기

앞에서는 개념을 단계별로 나누어 확인했습니다.

이 단계는 처음부터 복사해서 실행하기 위한 정답지가 아니라, **앞에서 배운 흐름이 어떻게 연결되는지 마지막에 다시 보는 요약**입니다.

In [ ]:
# 1. 데이터 준비
print("사용한 제목 수:", len(titles))

# 2. 분석 조건
print("사용 품사:", TARGET_TAGS)
print("최소 글자 길이:", MIN_LENGTH)
print("불용어:", STOPWORDS)

# 3. 최종 결과
print("최종 단어 수:", len(final_words))
print("빈도 합:", sum(final_counter.values()))
print("Top 10:", final_counter.most_common(10))

# 4. 저장한 결과 파일
print("상위 30개 CSV:", TOP30_PATH)
print("Word Cloud 이미지:", WORDCLOUD_PATH)

### 실습 29 결과 확인 및 정리

전체 흐름을 다시 정리하면 다음과 같습니다.

**CSV 불러오기 → 제목 준비 → Kiwi 형태소 분석 → 품사 필터 → 길이 필터 → 불용어 필터 → 전체 단어 추출 → Counter 빈도 계산 → Top 30 → 시각화 → 검증**

앞에서 각 단계를 이해한 뒤 이 흐름을 한 번에 보면 전체 분석 구조를 정리하기 쉽습니다.

## 실습 30. 오류가 발생했을 때 확인 순서

### AI에게 질문

> 다음 Python 코드를 실행했는데 오류가 발생했습니다.  
> 오류 원인을 초보자가 이해할 수 있게 설명한 뒤 현재 코드를 전부 다시 작성하지 말고 수정해야 할 부분만 최소한으로 알려 주세요.

오류가 발생하면 다음 순서로 확인합니다.

1. **어느 셀에서 처음 오류가 났는지 확인**
2. **Traceback의 마지막 오류 이름과 메시지 확인**
3. **파일 경로·컬럼 이름·설치된 라이브러리 확인**
4. **AI에게 실제 코드와 실제 오류 메시지를 함께 전달**

자주 볼 수 있는 예:

- `FileNotFoundError`
- `KeyError: '상품명'`
- `ModuleNotFoundError: No module named 'kiwipiepy'`
- `ModuleNotFoundError: No module named 'wordcloud'`

In [ ]:
# 경로/컬럼 문제를 확인할 때 사용할 수 있는 기본 점검 코드입니다.
print("CSV 파일 존재:", Path(DATA_PATH).exists())
print("현재 컬럼:", df_books.columns.tolist())
print("폰트 파일 존재:", FONT_PATH.exists())

## 실습 31. 결과가 이상할 때 확인할 것

코드가 오류 없이 실행되어도 결과가 이상할 수 있습니다.

### 상위 단어가 전부 의미 없어 보일 때

- 품사 필터가 적용되었는가?
- 최소 글자수 조건이 적용되었는가?
- 실제 결과를 보고 불용어를 추가할 필요가 있는가?

### 중요한 단어가 사라졌을 때

- `MIN_LENGTH`가 너무 큰가?
- 불용어에 잘못 포함되었는가?
- Kiwi가 예상과 다른 품사로 분석했는가?

### 영문 단어가 따로 집계될 때

- `.lower()`로 대소문자를 통일했는가?

### Word Cloud 한글이 깨질 때

- `FONT_PATH`가 실제 한글 폰트 파일을 가리키는가?

In [ ]:
# 특정 제목의 형태소 분석 결과를 직접 확인하는 방법입니다.
check_title = "확인하고 싶은 도서 제목"

for token in kiwi.tokenize(check_title):
    print(token.form, token.tag)

## 실습 32. 이번 Chapter의 분석 조건 기록하기

재현 가능한 분석을 위해 어떤 조건으로 결과를 만들었는지 Notebook에 남깁니다.

### 분석 조건

- 데이터: Chapter 01에서 전처리한 `book_bestseller_clean.csv`
- 분석 컬럼: `상품명`
- 형태소 분석기: Kiwi
- 사용 품사: NNG, NNP, SL
- 최소 단어 길이: 2글자
- 불용어: 세트, 개정판, 에디션, 리커버, 한정판
- 빈도 집계: `collections.Counter`
- 비교 방법: pandas `value_counts()`
- 시각화: 막대그래프, Word Cloud
- 상위 빈도 결과: 30개

이 조건을 기록해 두면 나중에 결과가 달라졌을 때 **어떤 전처리 조건이 바뀌었는지** 확인하기 쉽습니다.

## 실습 33. Notebook 최종 실행 확인

제출하거나 다음 Chapter로 넘어가기 전에 가능하면 다음 순서로 실행합니다.

**Kernel Restart → Run All**

처음부터 마지막까지 순서대로 다시 실행했을 때 오류가 없어야 합니다.

### 확인 체크리스트

- [ ] `book_bestseller_clean.csv`를 정상적으로 불러왔다.
- [ ] `상품명` 컬럼을 확인했다.
- [ ] Dictionary로 빈도 누적 원리를 확인했다.
- [ ] Counter로 빈도를 계산했다.
- [ ] pandas `value_counts()` 결과도 확인했다.
- [ ] 단순 토큰화의 한계를 확인했다.
- [ ] Kiwi 형태소 분석 결과를 직접 확인했다.
- [ ] 사용할 품사를 명시했다.
- [ ] 최소 글자수 조건을 적용했다.
- [ ] 불용어 기준을 기록했다.
- [ ] 상위 30개 단어를 확인했다.
- [ ] `chapter02_top30_words.csv`를 저장했다.
- [ ] Word Cloud를 생성했다.
- [ ] `chapter02_wordcloud.png`를 저장했다.
- [ ] 상위 단어를 원본 제목에서 확인했다.
- [ ] 실제 결과를 바탕으로 Markdown을 작성했다.

## 실습 34. 이번 Chapter에서 꼭 기억할 개념

### 1. 빈도 분석은 같은 값을 세는 작업에서 시작합니다

단어가 등장할 때마다 1씩 증가시키면 단어별 등장 횟수를 계산할 수 있습니다.

### 2. Dictionary, Counter, value_counts()는 연결된 개념입니다

- Dictionary → 원리 이해
- Counter → Python에서 간단한 빈도 집계
- value_counts() → pandas Series의 간단한 빈도 집계

### 3. 한국어는 단순 공백 분리만으로 부족할 수 있습니다

형태소 분석을 이용하면 품사를 확인하고 명사 중심으로 단어를 추출할 수 있습니다.

### 4. 전처리 조건은 결과를 바꿉니다

사용 품사, 최소 글자수, 불용어, 영문 대소문자 처리에 따라 상위 단어가 달라질 수 있습니다.

### 5. Word Cloud는 빈도를 시각적으로 표현합니다

큰 단어는 이번 조건에서 많이 등장했다는 뜻입니다.  
큰 단어가 판매 원인이나 독자 선호를 직접 의미하지는 않습니다.

## 실습 35. Chapter 02 결과물 정리

이번 Chapter를 끝내면 `notebooks/book-text-ml` 폴더에 다음 결과물이 있어야 합니다.

- `chapter02.ipynb`
- `chapter02_top30_words.csv`
- `chapter02_wordcloud.png`

Notebook 안에는 다음 내용이 남습니다.

- 데이터 불러오기
- Dictionary 빈도 집계 원리
- Counter 빈도 집계
- pandas `value_counts()` 비교
- 단순 토큰화
- Kiwi 품사 확인
- 명사 중심 단어 추출
- 최소 글자수 필터링
- 불용어 제거
- 상위 30개 빈도표
- 막대그래프
- Word Cloud
- 원본 데이터 검증
- 실제 결과를 이용한 분석 Markdown

다음 Chapter에서는 이번에 추출한 단어를 **머신러닝이 사용할 수 있는 숫자 벡터**로 바꾸는 과정을 학습합니다.